# Eco friendly arrays
Up to this point, there has been absolutely no concern regarding the memory allocation and memory was allocated wherever. However, with a little bit of changes, we can use some buffer arrays that can be allocated once and then rewritten every time. 

To demonstrate the idea we will start by rewriting the `E_bottoms_up` function to use these preallocated arrays instead of allocating at every call. We recall that this was defined as: 
```python
def E_bottoms_up(
    A: NDArray[float],
    a:float, 
    i_max:int,
    B: NDArray[float], 
    b:float, 
    j_max:int, 
    t_max: int, 
    out: Union[None, NDArray]=None) -> Union[NDArray[float],None]:
    """
    [...]
    """

    E_ab_t_array = np.zeros((i_max + 1, j_max + 1, t_max + 1))

    X_ab = B - A

    p = a + b
    mu = (a * b) / p
    X_pa = b / p * X_ab
    X_pb = -a / p * X_ab

    #Base case i = j = t = 0 (Helgaker 9.5.8)
    E_ab_t_array[0, 0, 0] = np.exp(-mu * X_ab**2)
    
    # Compute the [i,0,0] entries (Helgaker 9.5.20)
    for i in range(1, i_max+1):       
        E_ab_t_array[i, 0, 0] +=  X_pa * E_ab_t_array[i - 1, 0, 0]
        E_ab_t_array[i, 0, 0] += 1 / (2 * p) * ((i-1)* E_ab_t_array[i - 2, 0,0])

    # Compute the [0,j,0] entries (Helgaker 9.5.21)
    for j in range(1, j_max+1):
        E_ab_t_array[0, j, 0] +=  X_pb * E_ab_t_array[ 0, j - 1,0]
        E_ab_t_array[0, j, 0] += 1 / (2 * p) * ((j-1)* E_ab_t_array[ 0,j - 2,0])

    # Compute the [i,0,t] entries (Helgaker 9.5.18)
    for i in range(1,i_max+1):
        for t in range(1, t_max+1):
            E_ab_t_array[i,0,t] = (2*p)**(-t) * math.comb(i,t)* E_ab_t_array[i-t,0,0]
        
    # Compute the [0,j,t] entries (Helgaker 9.5.19)
    for j in range(1,j_max+1):
        for t in range(1, t_max+1):
            E_ab_t_array[0,j,t] = (2*p)**(-t) * math.comb(j,t)* E_ab_t_array[0,j-t,0]
    
    # Compute the [i, j, 0] entries (Helgaker 9.5.20)
    for i in range(1, i_max+1):
        for j in range(1, j_max+1):
            E_ab_t_array[i, j, 0] = X_pa * E_ab_t_array[i-1, j, 0] 
            E_ab_t_array[i, j, 0] += (1 / (2*p)) * ((i - 1) * E_ab_t_array[i-2, j, 0] + (j) * E_ab_t_array[i-1, j-1, 0])

    # Compute the rest entries (Helgaker 9.5.17)
    for i in range(1, i_max+1):
        for j in range(1,j_max+1):
            for t in range(1, t_max+1): 
                E_ab_t_array[i,j,t] = 1 / (2*p*t) * (i * E_ab_t_array[i-1,j,t-1] + j * E_ab_t_array[i,j-1,t-1])

    
    return E_ab_t_array
``` 

However, the price of using generalized arrays is that we require to treat them flatly. Therefore the functions defined to retrieve the flat indices will become quite handy here. 

In [1]:
from typing import Union 
import math

import numpy as np 
from numpy.typing import NDArray

from py_mods.src.integrals.internal.array_utils import * 
from py_mods.src.integrals.GTO import create_normalized_GTO, GTO, g_abcd_shell
from py_mods.src.integrals.internal.hermite_utils import R_tuv_n

In [2]:
def E_bottoms_up(
    A: NDArray[np.float32],
    a: float,
    i_max: int,
    B: NDArray[np.float32],
    b: float,
    j_max: int,
    t_max: int,
    out: Union[None, NDArray] = None,
) -> Union[NDArray[np.float32], None]:
    """
    [...]
    """

    if out is not None:
        E_ab_t_array = out
        E_ab_t_array[:] = 0
    else:
        E_ab_t_array = np.zeros([(i_max + 1) * (j_max + 1) * (t_max + 1)], dtype=np.float32)

    s_i = i_max + 1
    s_j = j_max + 1
    s_t = t_max + 1

    X_ab = B - A

    p = a + b
    mu = (a * b) / p
    X_pa = b / p * X_ab
    X_pb = -a / p * X_ab

    # Base case i = j = t = 0 (Helgaker 9.5.8)
    E_ab_t_array[IDX3DC(0, 0, 0, s_i, s_j, s_t)] = np.exp(-mu * X_ab**2)

    # Compute the [i,0,0] entries (Helgaker 9.5.20)
    for i in range(1, i_max + 1):
        E_ab_t_array[IDX3DC(i, 0, 0, s_i, s_j, s_t)] += (
            X_pa * E_ab_t_array[IDX3DC(i - 1, 0, 0, s_i, s_j, s_t)]
        )
        E_ab_t_array[IDX3DC(i, 0, 0, s_i, s_j, s_t)] += (
            1 / (2 * p) * ((i - 1) * E_ab_t_array[IDX3DC(i - 2, 0, 0, s_i, s_j, s_t)])
        )

    # Compute the [0,j,0] entries (Helgaker 9.5.21)
    for j in range(1, j_max + 1):
        E_ab_t_array[IDX3DC(0, j, 0, s_i, s_j, s_t)] += (
            X_pb * E_ab_t_array[IDX3DC(0, j - 1, 0, s_i, s_j, s_t)]
        )
        E_ab_t_array[IDX3DC(0, j, 0, s_i, s_j, s_t)] += (
            1 / (2 * p) * ((j - 1) * E_ab_t_array[IDX3DC(0, j - 2, 0, s_i, s_j, s_t)])
        )

    # Compute the [i,0,t] entries (Helgaker 9.5.18)
    for i in range(1, i_max + 1):
        for t in range(1, min(i + 1, t_max + 1)):
            E_ab_t_array[IDX3DC(i, 0, t, s_i, s_j, s_t)] = (
                (2 * p) ** (-t)
                * math.comb(i, t)
                * E_ab_t_array[IDX3DC(i - t, 0, 0, s_i, s_j, s_t)]
            )

    # Compute the [0,j,t] entries (Helgaker 9.5.19)
    for j in range(1, j_max + 1):
        for t in range(1, min(j + 1, t_max + 1)):
            E_ab_t_array[IDX3DC(0, j, t, s_i, s_j, s_t)] = (
                (2 * p) ** (-t)
                * math.comb(j, t)
                * E_ab_t_array[IDX3DC(0, j - t, 0, s_i, s_j, s_t)]
            )

    # Compute the [i, j, 0] entries (Helgaker 9.5.20)
    for i in range(1, i_max + 1):
        for j in range(1, j_max + 1):
            E_ab_t_array[IDX3DC(i, j, 0, s_i, s_j, s_t)] = (
                X_pa * E_ab_t_array[IDX3DC(i - 1, j, 0, s_i, s_j, s_t)]
            )
            E_ab_t_array[IDX3DC(i, j, 0, s_i, s_j, s_t)] += (1 / (2 * p)) * (
                (i - 1) * E_ab_t_array[IDX3DC(i - 2, j, 0, s_i, s_j, s_t)]
                + (j) * E_ab_t_array[IDX3DC(i - 1, j - 1, 0, s_i, s_j, s_t)]
            )

    # Compute the rest entries (Helgaker 9.5.17)
    for i in range(1, i_max + 1):
        for j in range(1, j_max + 1):
            for t in range(1, t_max + 1):
                E_ab_t_array[IDX3DC(i, j, t, s_i, s_j, s_t)] = (
                    1
                    / (2 * p * t)
                    * (
                        i * E_ab_t_array[IDX3DC(i - 1, j, t - 1, s_i, s_j, s_t)]
                        + j * E_ab_t_array[IDX3DC(i, j - 1, t - 1, s_i, s_j, s_t)]
                    )
                )

    return E_ab_t_array[: s_i * s_j * s_t].reshape(s_i, s_j, s_t)

In [3]:
def g_abcd_shell_new(
    gto1: GTO, gto2: GTO, gto3: GTO, gto4: GTO, k_hyper: int = 80
) -> NDArray[np.float64]:
    """
    Shell-based ERI tensor calculator. The idea here is that instead of
    working with projections as in the original g_abcd function, this
    function reuses intermediates. The E_AB coefficients are calculated
    once and then read from this, instead of recomputing them in the innermost
    loop.

    Recall that Gaussian overlap distributions over cartesian coordinates
    can be expanded as a LC of Hermite Gaussians with fixed Hermite polynomial
    coefficients E^{ab}_{t} which are much easier to manipulate. (Helgaker sec 9.5)

    Parameters
    ----------
    gto1, gto2, gto3, gto4 : GTO
        Gaussian primitive basis functions

    Returns
    -------
    NDArray[np.float64]
        ERI tensor for the given shells
    """

    a = gto1.exp
    b = gto2.exp
    c = gto3.exp
    d = gto4.exp

    r_A = gto1.R
    r_B = gto2.R
    r_C = gto3.R
    r_D = gto4.R

    p = a + b
    r_P = (a * r_A + b * r_B) / p

    q = c + d
    r_Q = (c * r_C + d * r_D) / q

    r_PQ = r_P - r_Q

    alpha = p * q / (p + q)

    # We compute over + 1 dimension due to the way the recurrent function works.
    t_max = u_max = v_max = gto1.total_L + gto2.total_L + 1
    tau_max = nu_max = phi_max = gto3.total_L + gto4.total_L + 1

    # Compute the hermite auxiliary integral
    Hermite_integral = R_tuv_n(
        alpha, r_PQ, t_max + tau_max, u_max + nu_max, v_max + phi_max, k_hyper
    )

    array1 = np.zeros([1024], dtype=np.float32)
    array2 = np.zeros([1024], dtype=np.float32)
    array3 = np.zeros([1024], dtype=np.float32)

    # Compute coefficients of A and B for each cartesian projection
    E_AB_x = E_bottoms_up(
        r_A[0], a, gto1.total_L, r_B[0], b, gto2.total_L, t_max, out=array1
    )
    E_AB_y = E_bottoms_up(
        r_A[1], a, gto1.total_L, r_B[1], b, gto2.total_L, u_max, out=array2
    )
    E_AB_z = E_bottoms_up(
        r_A[2], a, gto1.total_L, r_B[2], b, gto2.total_L, v_max, out=array3
    )

    E_AB_full = np.zeros((gto1.l_dim, gto2.l_dim, t_max, u_max, v_max))

    # Since there are different projections, the hole AB tensor for each projection pair
    # is obtained by contracting over the angular momenta of each projection in each
    # cartesian coordinate.
    for p1, proj_1 in enumerate(gto1.l_projections):
        for p2, proj_2 in enumerate(gto2.l_projections):
            i, k, m = proj_1
            j, l, n = proj_2

            # This way, to each projection vector, there is associated an index p1 or p2.
            # Then t, u, v indicate the degree of the hermite polynomial t <= i+j...
            # By contracting this way, the individual angular momenta indices are removed,
            # and only the total Hermite coefficient between two projections with specific
            # t,u,v is stored.
            E_AB_full[p1, p2, :, :, :] = np.einsum(
                "t, u, v -> tuv",
                E_AB_x[i, j, :t_max],
                E_AB_y[k, l, :u_max],
                E_AB_z[m, n, :v_max],
            )

    # This is repeated for C and D
    E_CD_x = E_bottoms_up(
        r_C[0], c, gto3.total_L, r_D[0], d, gto4.total_L, tau_max, out=array1
    )
    E_CD_y = E_bottoms_up(
        r_C[1], c, gto3.total_L, r_D[1], d, gto4.total_L, nu_max, out=array2
    )
    E_CD_z = E_bottoms_up(
        r_C[2], c, gto3.total_L, r_D[2], d, gto4.total_L, phi_max, out=array3
    )

    E_CD_full = np.zeros((gto3.l_dim, gto4.l_dim, tau_max, nu_max, phi_max))
    for p3, proj_3 in enumerate(gto3.l_projections):
        for p4, proj_4 in enumerate(gto4.l_projections):
            ii, kk, mm = proj_3
            jj, ll, nn = proj_4
            E_CD_full[p3, p4, :, :, :] = np.einsum(
                "t, u, v -> tuv",
                E_CD_x[ii, jj, :tau_max],
                E_CD_y[kk, ll, :nu_max],
                E_CD_z[mm, nn, :phi_max],
            )

    eri_block = np.zeros((gto1.l_dim, gto2.l_dim, gto3.l_dim, gto4.l_dim))

    # Now the eri tensor is defined as (Helgaker 9.9.33):
    # factor * sum_{t,u,v,tau,nu,phi} E^{AB}_{t,u,v} E^{CD}_{tau,nu,phi} * R_{..}(alpha, R_{PQ})
    for t in range(t_max):
        for u in range(u_max):
            for v in range(v_max):
                for tau in range(tau_max):
                    for nu in range(nu_max):
                        for phi in range(phi_max):
                            sign = (-1.0) ** (tau + nu + phi)
                            integral = (
                                Hermite_integral[t + tau, u + nu, v + phi, 0] * sign
                            )
                            eri_block += integral * np.einsum(
                                "ij,kl->ijkl",
                                E_AB_full[:, :, t, u, v],
                                E_CD_full[:, :, tau, nu, phi],
                            )

    # And the normalization constants are built as a tensor product of the normalization
    # Constant vectors
    norm_tensor = np.einsum(
        "i,j,k,l->ijkl",
        gto1.normalization_constants,
        gto2.normalization_constants,
        gto3.normalization_constants,
        gto4.normalization_constants,
    )

    # And normalization happens as a elementwise product
    eri_block *= norm_tensor

    # And the factor (Helgaker 9.9.33) is applied here
    return 2 * np.power(np.pi, 2.5) / (p * q * np.sqrt(p + q)) * eri_block

In [4]:
l = 0

r1 = np.array([0.0, 0.0, 0.0])
r2 = np.array([0.0, 0.0, 1.4])

# STO-3G-like exponents for H
H_exps = 3.42525091
H_coeffs = 0.15432897

H_1s_1 = create_normalized_GTO(r1, H_exps, l)
H_1p_1 = create_normalized_GTO(r1, H_exps, l + 1)
H_1d_1 = create_normalized_GTO(r2, H_exps, l + 3)
H_1s_2 = create_normalized_GTO(r2, H_exps, l)
H_1p_2 = create_normalized_GTO(r2, H_exps, l + 1)
H_1d_2 = create_normalized_GTO(r2, H_exps, l + 3)

In [5]:
%%timeit
eri1 = g_abcd_shell_new(H_1d_1, H_1p_1, H_1s_2, H_1d_2)

22.3 ms ± 314 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [6]:
%%timeit
eri2 = g_abcd_shell(H_1d_1, H_1p_1, H_1s_2, H_1d_2)

22.4 ms ± 772 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [7]:
eri1 = g_abcd_shell_new(H_1d_1, H_1p_1, H_1s_2, H_1d_2)
eri2 = g_abcd_shell(H_1d_1, H_1p_1, H_1s_2, H_1d_2)

np.allclose(eri1, eri2)

True

Which at this point is slower, but it is expected to be faster once compiled. 